# From Screaming Frog custom extraction to an NLP-ready dataset

Companion notebook for the MLforSEO guide **"Web scraping of page content in 3 easy steps with Screaming Frog"** by Lazarina Stoy.

The scraping itself is code-free (that's the point of the guide). This notebook helps with the two bits around it:

1. **Test a CSS selector before you crawl.** Paste a few URLs and the selector you copied from Chrome DevTools; see exactly what text it would extract. If it returns nothing, fix the selector now, not after a 10,000-URL crawl.
2. **Turn the Screaming Frog export into a clean dataset** for NLP and machine learning: merge multi-match columns (`Content 1`, `Content 2`...), clean whitespace, drop empty and thin pages, remove exact duplicates, add word counts, and save as CSV and JSONL.

Section 1 also works as a tiny, polite scraper for a handful of URLs if you don't have a Screaming Frog licence yet. For anything bigger, use the SEO Spider.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q beautifulsoup4
import re, json, time, hashlib
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from urllib import robotparser

## 1. Test your CSS selector

Use the selector exactly as you'd paste it into Screaming Frog's *CSSPath* field. Tip: Chrome's *Copy selector* often gives you a long, position-based path (`body > div:nth-child(3) > ...`). A short class-based selector such as `.entry-content` or `article` is usually more robust across templates.

In [ ]:
TEST_URLS = [
    "https://www.mlforseo.com/machine-learning-implementation-guides/keyword-research/pytrends-google-trends-api-python/",
    "https://www.mlforseo.com/machine-learning-tutorials/fuzzy-matching/what-fuzzy-matching-is-and-where-it-fits-in-seo/",
]
CSS_SELECTOR = ".entry-content"
USER_AGENT = "Mozilla/5.0 (compatible; MLforSEO selector tester)"
DELAY_SECONDS = 2   # be polite

def allowed_by_robots(url):
    p = urlparse(url)
    rp = robotparser.RobotFileParser(f"{p.scheme}://{p.netloc}/robots.txt")
    try:
        rp.read()
    except Exception:
        return True
    return rp.can_fetch(USER_AGENT, url)

def extract_text(url, selector):
    if not allowed_by_robots(url):
        return {"url": url, "status": "blocked by robots.txt", "matches": 0, "text": ""}
    r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
    if r.status_code != 200:
        return {"url": url, "status": r.status_code, "matches": 0, "text": ""}
    soup = BeautifulSoup(r.text, "html.parser")
    nodes = soup.select(selector)
    for n in nodes:                      # Screaming Frog's 'Extract Text' also ignores scripts and styles
        for t in n(["script", "style", "noscript"]):
            t.decompose()
    text = " ".join(n.get_text(" ", strip=True) for n in nodes)
    return {"url": url, "status": 200, "matches": len(nodes), "text": text}

results = []
for u in TEST_URLS:
    results.append(extract_text(u, CSS_SELECTOR))
    time.sleep(DELAY_SECONDS)

test = pd.DataFrame(results)
test["words"] = test["text"].str.split().str.len()
for _, r in test.iterrows():
    print(f"{r['url']}\n  status={r['status']} matches={r['matches']} words={r['words']}\n  preview: {r['text'][:200]}...\n")

**Reading the result:** `matches=0` means the selector finds nothing (wrong selector, or the content is loaded with JavaScript, in which case switch Screaming Frog to JavaScript rendering). `matches` above 1 means the selector hits several elements; Screaming Frog will put each one in its own column (`Content 1`, `Content 2`...), which section 2 merges back together.

Note that this test reads the raw HTML, like Screaming Frog's default text-only mode. If the text only appears in the rendered page, this test returns nothing even though a JavaScript-rendering crawl would work.

## 2. Clean the Screaming Frog export

In the SEO Spider, open the **Custom Extraction** tab and click **Export** (or use your saved export). Upload the CSV to Colab (folder icon on the left) and set the file name and the name you gave your extractor.

No export yet? Leave `SF_EXPORT_CSV = None` and the notebook uses the results of section 1 so you can see how it works.

In [ ]:
SF_EXPORT_CSV = None            # e.g. "custom_extraction_all.csv"
EXTRACTOR_NAME = "Content"      # the name you typed in Screaming Frog (columns become "Content 1", "Content 2"...)
MIN_WORDS = 50                  # drop thinner pages

if SF_EXPORT_CSV:
    raw = pd.read_csv(SF_EXPORT_CSV)
else:
    raw = test.rename(columns={"url": "Address", "text": f"{EXTRACTOR_NAME} 1"})[["Address", f"{EXTRACTOR_NAME} 1"]]

extract_cols = [c for c in raw.columns if re.fullmatch(re.escape(EXTRACTOR_NAME) + r" \d+", c)]
print("Extractor columns found:", extract_cols)

def clean(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text

df = pd.DataFrame({"url": raw["Address"]})
df["text"] = raw[extract_cols].fillna("").astype(str).agg(" ".join, axis=1).map(clean)
df["words"] = df["text"].str.split().str.len()
df["hash"] = df["text"].map(lambda t: hashlib.md5(t.lower().encode()).hexdigest())

before = len(df)
df = df[df["words"] >= MIN_WORDS]
thin = before - len(df)
dupes = df.duplicated("hash").sum()
df = df.drop_duplicates("hash").drop(columns="hash").reset_index(drop=True)
print(f"{before} rows -> {len(df)} kept ({thin} under {MIN_WORDS} words, {dupes} exact duplicates removed)")
df.head()

## 3. Save the dataset

CSV for spreadsheets and Data Studio, JSONL (one page per line) for most NLP and embedding tools.

In [ ]:
df.to_csv("page_content_dataset.csv", index=False)
with open("page_content_dataset.jsonl", "w", encoding="utf-8") as f:
    for rec in df.to_dict("records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("Saved page_content_dataset.csv and page_content_dataset.jsonl")
print(df["words"].describe().round(0).to_string())

if "google.colab" in sys.modules:
    from google.colab import files
    files.download("page_content_dataset.csv")

## What next?

This dataset is the input for most MLforSEO tutorials: topic modelling (LDA, BERTopic), entity extraction, sentiment analysis, embeddings and internal link audits. See the [MLforSEO code hub](https://github.com/lazarinastoy/mlforseo-code).